In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========
# 练习目标：抓取食谱列表页 → 用 Chat Completions 随机挑一道菜，并生成常规/空气炸锅做法（Markdown）
# 和本课 Day 1 关系：BeautifulSoup 清洗 HTML + system/user messages + OpenAI SDK
# 怎么跑：准备好 .env（OPENAI_API_KEY）后，从上到下 Shift+Enter

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入 requests：用 HTTP GET 下载目标网页
import requests

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从 bs4 导入 BeautifulSoup：解析 HTML，去掉无关标签后抽出纯文本
from bs4 import BeautifulSoup
# 从 IPython.display 导入 Markdown：在笔记本里渲染模型返回的 Markdown
from IPython.display import Markdown
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [ ]:
# ========== 抓取工具：带浏览器 UA 的 GET + BeautifulSoup 清洗 ==========

# 浏览器风格 User-Agent：很多网站会拒「无头」默认 UA；字符串保持原样（影响请求是否成功）
# Standard headers to fetch a website — 可运行字符串/键名不改
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """抓取 url 对应页面：返回「标题 + 正文」纯文本（本实现不做字符截断）。"""
    # GET 目标页；带上 headers 降低被拒概率
    response = requests.get(url, headers=headers)
    # 用 html.parser 把响应字节解析成可查询的 DOM
    soup = BeautifulSoup(response.content, "html.parser")
    # 取 <title> 文本；没有标题就用占位英文串（下游 prompt 仍可读）
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        # 删掉 script/style/img/input 等对摘要无用的节点，减少噪音
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 从 body 抽出可见文本；换行分隔、去掉首尾空白
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        # 没有 <body> 时退回空正文
        text = ""
    # 拼成「标题空行正文」；注意：这里没有 [:2000] 截断，整段交给模型
    return (title + "\n\n" + text)


In [ ]:
# ========== 环境：加载 .env 并粗查 OPENAI_API_KEY ==========

# 加载名为 .env 的文件到环境变量；override=True 表示用文件值覆盖已存在的同名变量
load_dotenv(override=True)
# 从环境变量取出 API Key，便于做形态检查（真正发请求时 OpenAI() 也会自己读）
api_key = os.getenv('OPENAI_API_KEY')

# 粗查密钥：是否存在、是否像 sk-proj- 项目密钥、是否首尾有空白
# Check the key — 下面 print 文案保持英文原样（排错指引，不翻译）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== 抓取示例页：Allrecipes「5 食材简易晚餐」列表 ==========

# URL 保持原样（影响抓取目标）；把清洗后的标题+正文存进 website，供后面拼进 user message
website = fetch_website_contents("https://www.allrecipes.com/easy-5-ingredient-dinners-11799627")
# 先肉眼看一眼抓到的文本噪音是否过大（导航、广告文案等）
print(website)


In [ ]:
# ========== system prompt：规定「厨艺助手」角色与输出结构 ==========
# 发给模型的指令字符串必须保留英文原文——改译会改变回答风格/行为

# Define System Prompt
system_prompt = """
You are a well-spoken cooking assistant that analyzes the contents of a food recipe website's meal list,
and finds a random recipe from within that list of recipes. You return the recipe name as a header, and if you cannot follow the link
use the internet to generate an appropriate recipe/dish summary or background in a well detailed paragraph.
Then you can return the list of ingredients followed by the cooking instructions for both conventional methods and air fryers 
(similarly, if you cannot follow the link, then use the internet to find or generate the appropriate ingredient list and instructions).
You do not need to select meals that include 'air fryer' in the name or already have air-fryer instructions. If a recipe does not have
air-fryer-related instructions, use the internet to find the best way to cook the recipe on an air fryer and generate the instructions.
Finally, display the information for that recipe. Format the ingredients (use this as a subheading) and instructions (use this as another subheading) 
into their own neatly organized tables.

You ignore text that might be website-navigation related that is not related to the link to the recipe.

You respond in markdown. Do not wrap the markdown in a code block!
"""


In [ ]:
# ========== user prompt 前缀：后面会拼接抓到的网页正文 ==========
# Define User Prompt — 英文原文保留（这是发给模型的 user 内容模板）

user_prompt_prefix = """
Here are the contents of a food recipe website's easy dinners list. Randomly choose one of the recipes. 
Return the recipe name, then provide a summary or background of the dish. Then return the ingredients, 
and cooking instructions for both conventional methods and air-frying.
"""


In [ ]:
# ========== 调用 Chat Completions：system + user，模型 gpt-5-nano ==========

# 创建默认 OpenAI 客户端；密钥从环境变量 OPENAI_API_KEY 自动读取
openai = OpenAI()

# messages：OpenAI 期望的列表结构——system 定规则，user 放任务+网页正文
messages = [
    {"role": "system", "content": system_prompt},
    # 前缀说明任务，再拼上 website（上一格抓到的文本）
    {"role": "user", "content": user_prompt_prefix + website}
]

# 发起一次非流式聊天；model id 字符串保持原样
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
# 取出助手回复文本，用 Markdown 在笔记本里渲染（不包代码块）
Markdown(response.choices[0].message.content)
